In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
# Carregar os dados
df = pd.read_csv('dataset_aluguel.csv')

In [ ]:
# Explorar os dados
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Valores ausentes
df.isnull().sum()

In [ ]:
df = df.dropna()

In [ ]:
# Remover outliers (IQR)
q1 = df['valor_aluguel'].quantile(0.25)
q3 = df['valor_aluguel'].quantile(0.75)
iqr = q3 - q1
df = df[(df['valor_aluguel'] >= q1 - 1.5 * iqr) & (df['valor_aluguel'] <= q3 + 1.5 * iqr)]
df.shape

In [ ]:
# Codificar as variáveis categóricas
df['localizacao_Periferia'] = df['localizacao_Periferia'].astype(int)
df['localizacao_Subúrbio'] = df['localizacao_Subúrbio'].astype(int)

In [ ]:
# Dividir em treino e teste
X = df.drop(columns=['id', 'valor_aluguel'])
y = df['valor_aluguel']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Pipeline com escalonamento e regressão linear
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', LinearRegression())
])
pipeline.fit(X_train, y_train)

In [ ]:
# Previsões
y_pred = pipeline.predict(X_test)

In [ ]:
# Métricas
print('R²:', r2_score(y_test, y_pred))
print('MAE:', mean_absolute_error(y_test, y_pred))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
# Resíduos
residuos = y_test - y_pred
plt.scatter(y_pred, residuos)
plt.axhline(0, color='red')
plt.xlabel('Previsto')
plt.ylabel('Resíduo')
plt.show()

In [ ]:
plt.hist(residuos)
plt.show()

In [ ]:
# Coeficientes
pd.Series(pipeline.named_steps['modelo'].coef_, index=X.columns)

## Coeficientes, limitações e melhorias

O tamanho do imóvel é a variável que mais pesa no aluguel, seguido do número de quartos. A idade da casa é a única com efeito negativo: quanto mais velha, menor o aluguel.

Limitações: a base tem só 100 imóveis, o que deixa o teste com 20 registros. Além disso, Periferia e Subúrbio aparecem com coeficiente positivo em relação à categoria de referência, o que não é o esperado e pode indicar que os dados são sintéticos ou que falta alguma variável.

Melhorias: usar mais dados, validação cruzada no lugar de uma única divisão de treino e teste, e testar modelos que capturem relações não lineares.